# Il blocco assemblato, e la cosa che lo rende impilabile

*Anatomia di un LLM* · capitolo 5, L'attenzione e il blocco · script `03-09`, `scripts/03_09_blocco_completo.py`

Attenzione piu' rete, ciascuna con la sua normalizzazione e la sua somma. La
somma e' il pezzo che conta: ogni blocco non riscrive quello che ha ricevuto,
ci aggiunge una correzione, e per questo se ne possono impilare quanti se ne
vuole.

Si esegue con `python3 scripts/03_09_blocco_completo.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/03_09_blocco_completo.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
import torch

from llm_anatomy.config import AgoGPTConfig
from llm_anatomy.model import Blk
from llm_anatomy.utils import (conta_parametri, prepara, stampa_shape,
                               tabella, titolo, verifica)

prepara()

cfg = AgoGPTConfig(d_model=192, n_head=6, block_size=64, vocab_size=93)
blocco = Blk(cfg).eval()
x = torch.randn(2, 32, cfg.d_model)

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


In [3]:
titolo("Il blocco, per intero")
stampa_shape("ingresso", x)
with torch.no_grad():
    y, _ = blocco(x)
stampa_shape("uscita", y)
print(f"  parametri del blocco: {conta_parametri(blocco)}")
for nome, modulo in (("attenzione", blocco.attn), ("FFN", blocco.ffn),
                     ("norme", blocco.norm1)):
    print(f"    {nome:<12} {conta_parametri(modulo)}")


Il blocco, per intero
---------------------
  ingresso                     (2, 32, 192)           float32
  uscita                       (2, 32, 192)           float32
  parametri del blocco: 442752
    attenzione   147456
    FFN          294912
    norme        192


In [4]:
titolo("Azzerandolo, non fa niente")
with torch.no_grad():
    blocco.attn.wo.weight.zero_()
    blocco.ffn.w_down.weight.zero_()
    y0, _ = blocco(x)
verifica(torch.allclose(y0, x, atol=1e-6),
         "azzerando le due proiezioni di uscita, l'ingresso attraversa intatto")
print("  non e' un caso di laboratorio: e' esattamente cio' che accade a")
print("  inizializzazione, dove quelle matrici sono scalate verso zero, e il")
print("  motivo per cui una pila profonda parte comunque stabile")


Azzerandolo, non fa niente
--------------------------
  [verifica superata] azzerando le due proiezioni di uscita, l'ingresso attraversa intatto
  non e' un caso di laboratorio: e' esattamente cio' che accade a
  inizializzazione, dove quelle matrici sono scalate verso zero, e il
  motivo per cui una pila profonda parte comunque stabile


In [5]:
titolo("Quanto scrive, rispetto a quello che riceve")
blocco = Blk(cfg).eval()
with torch.no_grad():
    a, _ = blocco.attn(blocco.norm1(x))
    dopo_attn = x + a
    f = blocco.ffn(blocco.norm2(dopo_attn))
    finale = dopo_attn + f

righe = [
    ["ingresso", f"{float(x.norm(dim=-1).mean()):.3f}".replace(".", ","), "---"],
    ["contributo attenzione", f"{float(a.norm(dim=-1).mean()):.3f}".replace(".", ","),
     f"{float(a.norm(dim=-1).mean() / x.norm(dim=-1).mean()):.3f}".replace(".", ",")],
    ["contributo FFN", f"{float(f.norm(dim=-1).mean()):.3f}".replace(".", ","),
     f"{float(f.norm(dim=-1).mean() / x.norm(dim=-1).mean()):.3f}".replace(".", ",")],
    ["uscita", f"{float(finale.norm(dim=-1).mean()):.3f}".replace(".", ","), "---"],
]
tabella(["quantita'", "norma media", "rispetto all'ingresso"], righe)
print("  a inizializzazione i contributi sono piccoli rispetto al flusso:")
print("  ogni blocco corregge, non riscrive")


Quanto scrive, rispetto a quello che riceve
-------------------------------------------
quantita'              norma media  rispetto all'ingresso
---------------------  -----------  ---------------------
ingresso                    13,783                    ---
contributo attenzione        1,510                  0,110
contributo FFN               1,427                  0,104
uscita                      13,936                    ---
  a inizializzazione i contributi sono piccoli rispetto al flusso:
  ogni blocco corregge, non riscrive


In [6]:
titolo("Entra e esce quasi uguale")
coseno = torch.nn.functional.cosine_similarity(x, finale, dim=-1).mean()
print(f"  coseno medio: {coseno:.4f}")
verifica(coseno > 0.9,
         "l'uscita del blocco resta quasi allineata al suo ingresso")
print("  e' il motivo per cui ha senso parlare di direzioni nel residual")
print("  stream e leggerle con la testa di uscita, come fa il logit lens")


Entra e esce quasi uguale
-------------------------
  coseno medio: 0.9875
  [verifica superata] l'uscita del blocco resta quasi allineata al suo ingresso
  e' il motivo per cui ha senso parlare di direzioni nel residual
  stream e leggerle con la testa di uscita, come fa il logit lens


In [7]:
titolo("Le due disposizioni, davanti a numeri grandi")
# Con un ingresso standard le due varianti si somigliano. La differenza si vede
# moltiplicando per cinque la scala del flusso in entrata: il pre-norm la
# conserva, il post-norm la cancella, perche' l'ultima operazione del blocco e'
# una normalizzazione.
torch.manual_seed(1337)
pre = Blk(cfg, pre_norm=True).eval()
torch.manual_seed(1337)
post = Blk(cfg, pre_norm=False).eval()

righe = []
for fattore in (1.0, 5.0, 20.0):
    xs = x * fattore
    with torch.no_grad():
        y_pre, _ = pre(xs)
        y_post, _ = post(xs)
    righe.append([
        f"{fattore:.0f}x",
        f"{float(xs.norm(dim=-1).mean()):.2f}".replace(".", ","),
        f"{float(y_pre.norm(dim=-1).mean()):.2f}".replace(".", ","),
        f"{float(y_post.norm(dim=-1).mean()):.2f}".replace(".", ","),
    ])
tabella(["scala dell'ingresso", "norma in entrata", "uscita pre-norm",
         "uscita post-norm"], righe)

with torch.no_grad():
    n1, _ = post(x)
    n2, _ = post(x * 20)
verifica(abs(float(n1.norm(dim=-1).mean()) - float(n2.norm(dim=-1).mean())) < 0.5,
         "l'uscita post-norm ha la stessa norma qualunque sia la scala in entrata")
print("  il post-norm consegna al blocco successivo un flusso sempre della")
print("  stessa taglia: comodo per la stabilita' numerica, e fatale per il")
print("  gradiente, perche' il percorso diretto dall'ingresso all'uscita viene")
print("  interrotto a ogni blocco. Lo misura lo script 03-08")


Le due disposizioni, davanti a numeri grandi
--------------------------------------------
scala dell'ingresso  norma in entrata  uscita pre-norm  uscita post-norm
-------------------  ----------------  ---------------  ----------------
1x                              13,78            13,95             13,86
5x                              68,92            68,94             13,86
20x                            275,66           275,65             13,86
  [verifica superata] l'uscita post-norm ha la stessa norma qualunque sia la scala in entrata
  il post-norm consegna al blocco successivo un flusso sempre della
  stessa taglia: comodo per la stabilita' numerica, e fatale per il
  gradiente, perche' il percorso diretto dall'ingresso all'uscita viene
  interrotto a ogni blocco. Lo misura lo script 03-08


## Da provare

azzera una sola delle due proiezioni di uscita invece di entrambe.